In [ ]:
# Low-RAM clone: run from the repo root (data/, runs/, experiments/ and the ember package are resolved from there).
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ember").is_dir() and _root != _root.parent:
    _root = _root.parent
assert (_root / "ember").is_dir(), "repo root (the folder with ember/) not found above the cwd"
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"cwd: {_root}")

# Ancient Rome, variable-width Sparse MF, variant `m3`: §3.1 Sparse MF → §3.2 concept features (whitened, Gemma)

> **Concept:** Ancient Rome · **model:** `google/gemma-2-2b-it` · **variant:** `m3` (entropy-allocated widths: m_i from the effective number of above-noise entries of each row).
>
> This is a clone of `low_ram_demands/ancient_rome_whitened.ipynb` (background-whitened + target-sharpened, low-RAM). It keeps
> **only steps 3.1 (factorization) and 3.2 (feature identification)**. Step 3.3 (erasure), 3.4 (δ tuning) and the analysis cells
> that need the erased or HF model are removed. The source notebook's WTA fit is replaced by the shared variable-width
> implementation in **`experiments/varwidth/`** (`core.fit_varwidth`, `common.VWConfig`). That is the same code, with the same
> settings, as the experiments in `results/varwidth/`, which `experiments/varwidth/run.py` runs on the matrices from
> `experiments/varwidth/prep.py`.
>
> **Hyperparameters.** `CFG = VWConfig(method="m3", refit=False)`. All other fields keep their defaults:
> k = 100; sparsity s = 0.01 (total budget K = ⌈s·|V*|⌉·k non-zeros); floor 0.002 and cap 0.05 (m_min = ⌈0.002·|V*|⌉,
> m_max = ⌈0.05·|V*|⌉; `fixed` ignores them); ridge 1e-4; max_iter 20,000; tol 1e-4; patience 200; churn_tol 0.005; init randn;
> m3_every 20; m3_window 0.02; m3_c 2.0. **Refit is off**, so Y = Y_dense ⊙ S, as in the paper. Seed 42.
> Whitening: BACKGROUND `"wiki"`, TARGET `"page"`, GAMMA 1.0, and ETA 0.05 / EPS_REL 1e-4 from
> `experiments/whitening/data/gemma/defaults.json`. Judge: τ = 2.0, confidence ≥ 0.85, 20 tokens per feature,
> `models/gemini-3.5-flash-lite`.
>
> - **3.1.c′** builds the transform `T`. `A` = Xᵀ (whitened) and `A0` = E_{V*}ᵀ (raw) are built exactly the way `prep.py`
>   builds `results/varwidth/prep/Ancient_Rome_whitened_S300.pt` and `_plain_S300.pt` (the end of 3.1.c′ compares them when those
>   files exist).
> - **3.1.d** fits `Z`, `Y` on `A` with `fit_varwidth(A, CFG, seed=SEED)`, cached in `RUN_DIR/fit.pt`. **3.1.d′** fits `Z0`, `Y0`
>   on `A0` with the same config (`RUN_DIR/fit0.pt`). **3.1.d″** shows the per-feature widths of both fits.
> - **3.2** selects $F_C$ from the whitened fit with the two-stage Gemini judge. **No Gemini calls are made for the original fit.**
>   **3.2.e′** lists its ρ ≥ τ candidates (`F_C0_rho`, each with its width and top-15 tokens) so you can compare them with $F_C$ by eye.
>   **3.2.w** uses those candidates where the source notebook used the judged `F_C0`.
>
> **Runs.** Caches go to `RUN_DIR = runs/variable_width/Ancient_Rome/m3/` (`fit.pt`, `fit0.pt`, `judge.json`,
> `summary.json`), so nothing collides with the source notebook's `runs/whiten/` caches.

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This notebook runs the first two stages of §3 for one concept.
Each stage is split into small blocks, and each block starts with a note on what it does.

| Step | Paper section | What happens | Code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $X^\top \approx ZY$ with a sparse $Y$ (variable width per feature) | `experiments/varwidth/core.py` `fit_varwidth` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |

**Notation.** `Z` $\in \mathbb R^{d\times k}$ and `Y` $\in \mathbb R^{k\times|V^*|}$ use the paper's shapes.

**Requirements:** a CPU is enough. You need a Gemini API key for Step 3.2 (whitened fit only).

---
## 0. Setup

### 0.1 Hyperparameters

| Symbol | Value | Source |
|---|---|---|
| variant | `m3`: `VWConfig(method="m3", refit=False)` | `experiments/varwidth/common.py` |
| $k$ (rank) | 100 (Gemma-2-2B-it) | App. A.1 |
| $s$ (budget) | 0.01: $K = \lceil s\,|V^*| \rceil\cdot k$ non-zeros in $Y$ in total | App. A.1 / `VWConfig.sparsity` |
| floor / cap | $m_{\min} = \lceil 0.002\,|V^*|\rceil$, $m_{\max} = \lceil 0.05\,|V^*|\rceil$ tokens per feature (not used by `fixed`) | `VWConfig` |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol / churn | 20,000 iters / 200 iters / $10^{-4}$ / 0.005 | `VWConfig` |
| refit | off ($Y = Y_{\text{dense}}\odot S$, as in the paper) | `VWConfig.refit` |
| seed | 42 | |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| whitening | background `wiki`, target `page`, $\gamma = 1$, $\eta = 0.05$, eps_rel $= 10^{-4}$ | `experiments/whitening/data/gemma/defaults.json` |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |

In [ ]:
import dataclasses
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

from experiments.varwidth.common import VWConfig

load_dotenv()  # GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to analyse ----------------------------------------------------
CONCEPT = "Ancient Rome"
MODEL_NAME = "google/gemma-2-2b-it"     # the varwidth experiments are Gemma only
SEED = 42

# ======== VARIANT: variable-width sparse MF (experiments/varwidth; the configs of the runs in results/varwidth/) ========
VARIANT = "m3"
VARIANTS = {
    "fixed1": VWConfig(method="fixed", refit=False),                 # the paper: each feature keeps its top ceil(0.01|V*|) tokens
    "fixed2": VWConfig(method="fixed", refit=False, sparsity=0.02),  # the same WTA rule at 2% per feature (twice the non-zeros)
    "m1":     VWConfig(method="m1", refit=False),                    # contribution pooling: global top-K of |y_ij|*||z_i||, floor/cap
    "m3":     VWConfig(method="m3", refit=False),                    # entropy-allocated widths: m_i ~ exp(H) of the above-noise top entries
    "rn":     VWConfig(method="rn", refit=False),                    # pooling on |y_ij| / ||y_i||_2 (row-normalized), floor/cap
}
# All other VWConfig fields stay at their defaults: rank 100, sparsity 0.01 (fixed2: 0.02), floor 0.002, cap 0.05, ridge 1e-4,
# max_iter 20000, tol 1e-4, patience 200, churn_tol 0.005, init randn, m3_every 20, m3_window 0.02, m3_c 2.0.
# =======================================================================================================================

# ---- Step 3.2: feature selection (App. A.3; = experiments/varwidth/select.py) ----
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite
JUDGE_FORMAT = "tokens"   # Stage-1 judge input: "tokens" (bare tokens, paper; what the experiments use)
                          # | "words" (token → the words it was cut from in S_C ∪ S_N, alphabetical)

# ---- background whitening + target sharpening (Instructions.txt, experiments/whitening/sharpen.py) ----
import re
import sys

sys.path.insert(0, "experiments/whitening")
from sharpen import BackgroundWhitener, Sharpener, Transform, principal_angle_sines

BACKGROUND = "wiki"      # the experiments' prep (experiments/varwidth/prep.py) uses "wiki"
TARGET = "page"          # ... and "page"
GAMMA = 1.0              # ... and gamma = 1
MODEL_SLUG = "gemma"
SMOKE = False            # True: quick pass (50-iteration fits, 3 judged features) into RUN_DIR + "_smoke"

# ---- overrides (set in an injected first cell, e.g. _OVERRIDES = {"VARIANT": "m1"}) ----
globals().update(globals().get("_OVERRIDES", {}))
assert VARIANT in VARIANTS, f"VARIANT must be one of {list(VARIANTS)}"
CFG = VARIANTS[VARIANT]
if SMOKE:
    CFG = dataclasses.replace(CFG, max_iter=50, patience=5)

# The fit's hyperparameters, under the source notebook's names (downstream cells use them)
RANK = CFG.rank          # k
SPARSITY = CFG.sparsity  # s: total budget K = ceil(s |V*|) * k (fixed: exactly ceil(s |V*|) per feature)
RIDGE = CFG.ridge        # lambda

# Whitening must match the experiments exactly (experiments/varwidth/prep.py)
from experiments.varwidth import prep as vw_prep

assert (MODEL_NAME, BACKGROUND, TARGET, GAMMA) == (vw_prep.MODEL_NAME, vw_prep.BACKGROUND, vw_prep.TARGET, vw_prep.GAMMA), \
    "whitening settings differ from experiments/varwidth/prep.py"

WH_DIR = Path("experiments/whitening/data") / MODEL_SLUG
defaults = json.loads((WH_DIR / "defaults.json").read_text())   # prep.py reads the same file (no fallback)
print(f"whitening defaults from {WH_DIR / 'defaults.json'}: eta={defaults['eta']} eps_rel={defaults['eps_rel']}")
ETA = defaults["eta"]          # target shrinkage (0.05)
EPS_REL = defaults["eps_rel"]  # eps = EPS_REL * mean(background eigenvalues) (1e-4)

# ---- run directory: resume caches (new location, nothing shared with runs/whiten/) ----
concept_slug = re.sub(r"\W+", "_", CONCEPT).strip("_")
_smoke = "_smoke" if SMOKE else ""
RUN_DIR = Path("runs/variable_width") / concept_slug / f"{VARIANT}{_smoke}"   # fit.pt, fit0.pt, judge.json, summary.json
RUN_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path: Path, default):
    return json.loads(path.read_text()) if path.exists() else default


def save_json(path: Path, obj) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")   # write then rename, so a crash never leaves a half file
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    tmp.replace(path)


def invalidate(directory: Path, names) -> None:
    """Move stale cache files aside (<name>.stale) so they are recomputed."""
    for name in names:
        p = directory / name
        if p.exists():
            p.replace(p.with_name(p.name + ".stale"))
            print(f"  moved stale {p} -> {p.name}.stale")

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | variant={VARIANT} | seed={SEED}")
print(f"CFG = {CFG}")
print(f"background={BACKGROUND} | target={TARGET} | gamma={GAMMA:g} | eta={ETA:g} | eps_rel={EPS_REL:g} | "
      f"smoke={SMOKE} | run dir={RUN_DIR}")

### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini, and only for the whitened fit. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [ ]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

### 0.3 Repo imports

The selection uses the same building blocks as the paper's pipeline (`ember.interpret_features`, `ember.utils`). The
factorization comes from `experiments/varwidth/core.py`, the module that the experiments in `results/varwidth/` use.

In [ ]:
from llm_utils.activation_generator import extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset
from ember.timing import Timer
from ember.utils import (
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    get_special_token_ids,
    set_seed,
)
from experiments.varwidth.common import budgets, mask_from_row_widths
from experiments.varwidth.core import fit_varwidth

set_seed(SEED)
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU (the experiments fit on CPU too).
DEVICE = "cpu"
print(f"device: {DEVICE}")

---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(sparse coefficients)}$$

The sparsity constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only a few tokens of $V^*$
(the paper: 1% each; here the variant's allocation rule decides each feature's width, see 3.1.d). That is why a feature
can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. **Note:** for Gemma, TL folds the $\sqrt{d}$ embedding
multiplier into `W_E`, so the factorization runs in this *scaled* space. (No model is built here: 3.1–3.2 only need
`W_E` and the tokenizer.)

In [ ]:
# Low-RAM: steps 3.1-3.2 use only W_E and the tokenizer, so no TransformerLens model is built.
# W_E is read straight from the checkpoint and matches TL's: Gemma = HF embedding * sqrt(d_model), Llama = as stored;
# TL skips writing-weight centering for RMSNorm models. E_DTYPE is the dtype this notebook's TL model used.
import json
import math
from types import SimpleNamespace

from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError
from safetensors import safe_open

E_DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:  # Llama-3: TL also pads with eos
    tokenizer.pad_token = tokenizer.eos_token


def load_input_embeddings(model_name: str) -> torch.Tensor:
    """Reads only model.embed_tokens.weight from the checkpoint (no model is built)."""
    try:
        index = json.loads(Path(hf_hub_download(model_name, "model.safetensors.index.json")).read_text())
        shard = index["weight_map"]["model.embed_tokens.weight"]
    except EntryNotFoundError:  # single-file checkpoint
        shard = "model.safetensors"
    with safe_open(hf_hub_download(model_name, shard), framework="pt") as f:
        return f.get_tensor("model.embed_tokens.weight")


class TokenizerOnly:
    """The parts of a HookedTransformer that 3.1-3.2 use: tokenizer, to_tokens, to_str_tokens."""

    def __init__(self, tok):
        self.tokenizer = tok

    def to_tokens(self, prompts, padding_side="left"):
        self.tokenizer.padding_side = padding_side
        return self.tokenizer(prompts, return_tensors="pt", padding=True)["input_ids"]  # BOS added by the tokenizer

    def to_str_tokens(self, ids):
        return self.tokenizer.batch_decode(torch.as_tensor(ids).reshape(-1, 1), clean_up_tokenization_spaces=False)


tl_model = TokenizerOnly(tokenizer)  # keeps the old names, so the cells below run unchanged
act_generator = SimpleNamespace(model=tl_model, data_device="cpu")

E = load_input_embeddings(MODEL_NAME).to(E_DTYPE)  # (|V|, d)
if "gemma" in MODEL_NAME.lower():
    E.mul_(torch.tensor(math.sqrt(E.shape[1]), dtype=E_DTYPE))  # in place: no second (|V|, d) copy
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [ ]:
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [ ]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
_K, _lo, _hi = budgets(len(vstar_ids), CFG)
print(f"budget K = ceil(s*|V*|)*k = {_K} non-zeros  (fixed: {_K // RANK} per feature; m1/m3/rn: {_lo}..{_hi} per feature)")

### 3.1.c′ Background whitening and target sharpening

Instructions steps 1–4, in the row convention of the notebook (rows are tokens; `sharpen.py`):

$$U = (E_{V^*} - \mathbf 1\boldsymbol\mu_B^\top)\,W,\quad W = Q(\Lambda+\epsilon I)^{-1/2}Q^\top;\qquad
S_\eta = (1-\eta)\,\tfrac1m U_C^\top U_C + \eta I = P\,\mathrm{diag}(g)\,P^\top;\qquad X = U\,S_\eta^{\gamma/2}$$

1. **Background** ($\boldsymbol\mu_B$, $Q$, $\Lambda$ of $\Sigma_B$) is loaded from `WH_DIR/bg_<BACKGROUND>_stats.pt`. It was computed once per
   model, in the same space as `E` (TL `W_E`: Gemma rows already × $\sqrt d$; Llama unscaled). $\epsilon$ = `EPS_REL` × mean($\Lambda$).
   Check: $W\Sigma_B W = Q\,\mathrm{diag}\big(\lambda/(\lambda+\epsilon)\big)Q^\top \approx I$, except along the smallest
   eigenvalues, which $\epsilon$ deliberately damps.
2. **Whitening** centres on $\boldsymbol\mu_B$ (not on the mean of $V^*$), so the concept's shift away from the background stays in the data.
3. **Target**: the token ids of the concept's Wikipedia page (`WH_DIR/targets/<concept>.json`, key `TARGET`), whitened with the
   same $W$. The second moment is taken around 0, so it includes that shift. $g_j$ are **enrichment ratios** ($g_j > 1$: the target
   varies more than the background along $\mathbf p_j$); $\eta$ = `ETA` puts a floor under the directions the target does not use.
4. **Sharpening** with $S_\eta^{\gamma/2}$ applies one fixed map to every row of $V^*$, concept and neutral alike. `GAMMA = 0` skips it.

The combined map is `T` (`Transform`): `T.forward(rows)` $= (\text{rows}-\boldsymbol\mu_B)\,M$ with $M$ = `T.matrix` $= W S_\eta^{\gamma/2}$, and
`T.unmap(z)` $= z\,S_\eta^{-\gamma/2}W^{-1}$ maps directions back. `transform(rows)` is the float32 helper the analysis cells use.

**Diagnostics.** The largest enrichment ratios, and the contrastive-PCA directions $\mathbf p_1,\dots,\mathbf p_5$: the whitened
Wikipedia background token types (`bg_wiki_types.json`) projected on each, top 15 shown (`*` = in the target set). If they are
concept tokens, the signal is there before any factorization. With `GAMMA = 0` the same diagnostics use a $\gamma=1$ sharpener that is
not applied.

The last lines compare `A` and `A0` with the experiments' cached matrices (`results/varwidth/prep/Ancient_Rome_{whitened,plain}_S300.pt`,
built by `experiments/varwidth/prep.py`), when those files exist.

From here on `A` is $X^\top$; `E_vstar_orig` (= `E_vstar`) keeps the raw rows and `A0` the raw data matrix.

In [ ]:
# Instructions steps 1-4: background whitening W, target second moment S_eta, sharpening S_eta^(gamma/2). float64, CPU.
E_vstar_orig = E_vstar                                    # raw rows (Gemma: TL-scaled), for the original fit 3.1.d'
A0 = E_vstar.T.contiguous().to(DEVICE)                    # (d, |V*|) raw

# ---- step 1: background statistics (built once per model by experiments/whitening) ----
BG_PATH = WH_DIR / f"bg_{BACKGROUND}_stats.pt"
TARGETS_PATH = WH_DIR / "targets" / f"{concept_slug}.json"
for _p, _what in [(BG_PATH, f"background statistics ({BACKGROUND})"), (TARGETS_PATH, f"target token ids for {CONCEPT!r}")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} is missing: the {_what} must be built first (experiments/whitening)")
stats_bg = torch.load(BG_PATH, map_location="cpu")
missing_keys = {"mu", "lam", "Q"} - set(stats_bg)
assert not missing_keys, f"{BG_PATH} lacks {missing_keys}"
assert stats_bg["Q"].shape == (D_MODEL, D_MODEL), f"{BG_PATH}: Q is {tuple(stats_bg['Q'].shape)}, expected d = {D_MODEL}"
stats_bg = {k: (v.double() if torch.is_tensor(v) else v) for k, v in stats_bg.items()}
bw = BackgroundWhitener(stats_bg, EPS_REL)

lam_bg = stats_bg["lam"].clamp_min(0)
lam_desc = lam_bg.sort(descending=True).values
shrink = lam_desc / (lam_desc + bw.eps)                   # eigenvalues of W Sigma_B W
n95 = math.ceil(0.95 * len(lam_desc))
print(f"background {BACKGROUND}: n = {stats_bg.get('n', '?')} types, d = {D_MODEL}, eps = {EPS_REL:g} x mean(lam) = {bw.eps:.3e}")
print(f"W Sigma_B W ~ I: max |lam/(lam+eps) - 1| over the top 95% of directions = {(shrink[:n95] - 1).abs().max():.3e}, "
      f"overall = {(shrink - 1).abs().max():.3e} (the smallest eigenvalues are damped by eps on purpose)")
print(f"cond(Sigma_B) = {(lam_desc[0] / lam_desc[-1]).item() if lam_desc[-1] > 0 else float('inf'):.3e}   "
      f"cond(Sigma_B + eps I) = {((lam_desc[0] + bw.eps) / (lam_desc[-1] + bw.eps)).item():.3e}")

# ---- step 3: target rows, whitened ----
targets = json.loads(TARGETS_PATH.read_text())
assert TARGET in targets, f"{TARGETS_PATH} has keys {list(targets)}, not {TARGET!r}"
target_ids = sorted({int(t) for t in targets[TARGET] if 0 <= int(t) < VOCAB_SIZE} - set(special))
assert target_ids, f"{TARGETS_PATH}[{TARGET!r}] has no usable token ids"
E_target = E.index_select(0, torch.tensor(target_ids, device=E.device)).detach().double().cpu()   # (m, d), same space as E_vstar
U_target = bw.whiten(E_target)                                                                     # (m, d)
target_set, vstar_set = set(target_ids), set(vstar_ids)
_codes31 = build_token_label_codes(CONCEPT, dataset, tokenizer)
tc_ids = {t for t in vstar_ids if _codes31.get(t, 0) == 1}                                         # T_C (concept-only)
print(f"|target| = {len(target_ids)} ({TARGET})   in V*: {len(target_set & vstar_set)} / {len(vstar_ids)}   "
      f"concept-only T_C covered: {len(target_set & tc_ids)} / {len(tc_ids)}")

# ---- step 4: sharpening; T = whitening (+ sharpening) ----
sh = Sharpener(U_target, ETA, GAMMA) if GAMMA > 0 else None
sh_diag = sh if sh is not None else Sharpener(U_target, ETA, 1.0)   # gamma = 0: diagnostics only, never applied
T = Transform(bw, sh)


def transform(rows: torch.Tensor) -> torch.Tensor:
    """Rows (n, d) in the original (TL) space -> X-space: (rows - mu_B) W S_eta^(gamma/2), float32."""
    return T.forward(rows.detach().cpu()).float()


E_X = transform(E_vstar)                                  # (|V*|, d)
E_vstar_X = E_X                                           # survives 3.3.a; all analysis cells use it
A = E_X.T.contiguous().to(DEVICE)                         # (d, |V*|): 3.1.d fits on this
m_tilde = U_target.mean(0)                                # whitened target mean (the whitened shift Delta)
m_X = m_tilde if sh is None else m_tilde @ sh.Sp          # its image in X-space (= mean of transform(E_target))

# ---- checks ----
I_d = torch.eye(D_MODEL, dtype=torch.float64)
T_inv = bw.W_inv if sh is None else sh.Sm @ bw.W_inv      # row form: z = z~ @ T_inv
print(f"inverse: max |T.matrix @ (Sm @ W_inv) - I| = {(T.matrix @ T_inv - I_d).abs().max():.2e}")
print(f"row norms: raw mean {E_vstar.norm(dim=1).mean():.3f}   X-space mean {E_X.norm(dim=1).mean():.3f}   "
      f"cond(S_eta) = {(sh_diag.g[0] / sh_diag.g[-1]).item():.3e}   cond(T.matrix) = {torch.linalg.cond(T.matrix).item():.3e}")
del I_d

# ---- diagnostics: enrichment ratios ----
g_top = sh_diag.g[:10].tolist()
print(f"\nenrichment ratios g (eta = {ETA:g}; directions the target does not use sit at g = eta): top 10 = {', '.join(f'{v:.2f}' for v in g_top)}")
print(f"directions with g > 1: {int((sh_diag.g > 1).sum())} / {D_MODEL}   g > 2: {int((sh_diag.g > 2).sum())}   "
      f"stretch range applied: g^(gamma/2) in [{sh_diag.g[-1].item() ** (GAMMA / 2):.3f}, {sh_diag.g[0].item() ** (GAMMA / 2):.3f}]")

# ---- diagnostics: contrastive PCA on the whitened Wikipedia background types ----
N_CPCA, N_CPCA_TOK = 5, 15
WIKI_TYPES_PATH = WH_DIR / "bg_wiki_types.json"
if WIKI_TYPES_PATH.exists():
    wiki_ids = [int(t) for t in json.loads(WIKI_TYPES_PATH.read_text()) if 0 <= int(t) < VOCAB_SIZE]
    P5 = sh_diag.P[:, :N_CPCA]                                         # (d, 5)
    signs = torch.sign(m_tilde @ P5)                                   # orient each p_j towards the target mean
    signs[signs == 0] = 1.0
    M5 = bw.W @ (P5 * signs)                                           # whiten + project in one (d, 5) map
    scores = torch.cat([(E.index_select(0, torch.tensor(wiki_ids[i:i + 20_000], device=E.device)).detach().double().cpu()
                         - bw.mu) @ M5 for i in range(0, len(wiki_ids), 20_000)])   # (N_wiki, 5)
    cpca = {}
    for j in range(N_CPCA):
        top = torch.argsort(scores[:, j], descending=True)[:N_CPCA_TOK].tolist()
        toks = tokenizer.convert_ids_to_tokens([wiki_ids[i] for i in top])
        cpca[f"p{j + 1} (g={sh_diag.g[j]:.2f})"] = [f"{t}{' *' if wiki_ids[i] in target_set else ''}  {scores[i, j]:.2f}"
                                                   for t, i in zip(toks, top)]
    hit = {k: sum(s.split("  ")[0].endswith(" *") for s in v) for k, v in cpca.items()}
    print(f"\ncPCA: top {N_CPCA_TOK} of {len(wiki_ids)} whitened Wikipedia types on p_1..p_{N_CPCA} "
          f"(* = in the target set; per direction: {hit})")
    cpca_df = pd.DataFrame(cpca)
    cpca_df.index += 1
    display(cpca_df)
    del scores
else:
    print(f"\n{WIKI_TYPES_PATH} not found: cPCA diagnostic skipped")

print(f"\nA (transformed): {tuple(A.shape)}   A0 (raw): {tuple(A0.shape)}")

# ---- check against the experiments' matrices (experiments/varwidth/prep.py), when they have been built ----
# prep.py builds A the same way: E from the checkpoint x sqrt(d), V* from ConceptDataset(neutral_sample_seed=42),
# A_plain = E_V*^T, A_whitened = T.forward(E_V*)^T with the same bg_wiki stats, page target, eta, eps_rel and gamma.
for _matrix, _mine in [("whitened", A), ("plain", A0)]:
    _p = vw_prep.prep_path(CONCEPT, _matrix)
    if not _p.exists():
        print(f"prep check [{_matrix}]: {_p} not built, skipped")
        continue
    _prep = torch.load(_p, weights_only=False)
    _same_v = list(_prep["vstar_ids"]) == list(vstar_ids)
    _diff = (_prep["A"].float() - _mine.float().cpu()).abs().max().item() if _same_v else float("nan")
    _extra = (f"  eta={_prep['eta']} eps_rel={_prep['eps_rel']} n_target={_prep['n_target']} (here {len(target_ids)})"
              if _matrix == "whitened" else "")
    print(f"prep check [{_matrix}] {_p}: same V*: {_same_v}   max |A_prep - A| = {_diff:.2e}{_extra}")
    del _prep

### 3.1.d Fit variable-width Sparse MF (`fit_varwidth`)

The shared loop of `experiments/varwidth/core.py` (with $\lambda$ = `CFG.ridge`) runs these steps at every iteration:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y_{\text{dense}} \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A,\qquad
S \leftarrow \mathrm{Allocate}(Y_{\text{dense}}, Z),\qquad Y \leftarrow Y_{\text{dense}}\odot S$$

After that, the rows of $Y$ are scaled to unit $\ell_2$ norm and the scale is absorbed into $Z$'s columns. The init is the paper's
$\mathcal N(0,1)$ with seed `SEED`. The fit stops once the relative error has not improved by $\ge$ `tol` for `patience` = 200
iterations **and** the support churn has stayed below `churn_tol` = 0.5% over those same iterations; otherwise it stops at `max_iter`.

**Variant `m3`: entropy-allocated widths.** Every `m3_every` = 20 iterations each feature's width is recomputed.
Take the top $W = \lceil 0.02\,|V^*|\rceil$ entries of the dense row $|y_{i,:}|$ and subtract the noise floor $c\,\sigma_i$
($c$ = `m3_c` = 2, $\sigma_i = 1.4826\cdot\mathrm{MAD}$ of the row). Square what is left and normalize it to $p$, so that
$r_i = \exp H(p)$ is the effective number of entries above the noise. The $r_i$ are scaled to sum to
$K = \lceil 0.01\,|V^*|\rceil\cdot k$, clipped to $[m_{\min}, m_{\max}] = [\lceil 0.002\,|V^*|\rceil, \lceil 0.05\,|V^*|\rceil]$ and rounded,
keeping the total exactly $K$. Between updates, feature $i$ keeps its top $m_i$ entries by $|y_{ij}|$.

**Refit is off** (`refit=False`): the kept entries keep their dense values ($Y = Y_{\text{dense}}\odot S$), as in the paper.

`A` is $X^\top$ from 3.1.c′, so `Z` (d × k) and `Y` (k × |V*|) are the transformed factorization that everything downstream uses.
The fit dict (the `fit.pt` format of `experiments/varwidth/common.py`: `Z`, `Y`, `widths`, `rel_err_hist`, `churn`, …) is cached in
`RUN_DIR/fit.pt` with its configuration. A cache made with another configuration (or another $V^*$) is moved aside as `*.stale`,
along with `judge.json` and `summary.json`, and the fit reruns. It prints progress every 100 iterations.

In [ ]:
# Resume: RUN_DIR/fit.pt is loaded when its configuration matches; otherwise it (and the caches built on it) are moved aside.
FIT_PATH = RUN_DIR / "fit.pt"
fit_meta = {"model": MODEL_NAME, "vstar_ids": vstar_ids, "matrix": "whitened", "background": BACKGROUND, "target": TARGET,
            "gamma": GAMMA, "eta": ETA, "eps_rel": EPS_REL, "n_target": len(target_ids), "config": CFG.to_dict(), "seed": SEED}
fit = torch.load(FIT_PATH, weights_only=False) if FIT_PATH.exists() else None
if fit is not None and fit.get("meta") != fit_meta:
    print(f"{FIT_PATH} was made with another configuration: re-fitting")
    invalidate(RUN_DIR, ["fit.pt", "judge.json", "summary.json"])
    fit = None
if fit is not None:
    print(f"loaded {FIT_PATH}")
else:
    fit = fit_varwidth(A, CFG, seed=SEED, device=DEVICE, verbose=True)
    fit["meta"] = fit_meta
    torch.save(fit, FIT_PATH)

Z = fit["Z"].float().cpu()          # (d, k)      X-space features z~_i (columns)
Y = fit["Y"].float().cpu()          # (k, |V*|)   sparse coefficients, unit-norm rows, one width per feature
widths = fit["widths"].long()       # (k,)        |S_i| = number of tokens feature i keeps
ridge_used = CFG.ridge              # fit_varwidth always uses CFG.ridge (no x10 retry as in fit_with_ridge)
t_fit = {"elapsed": fit["elapsed"]}
print(f"\nfitted in {t_fit['elapsed']:.1f}s: {fit['n_iter']} iterations, stop={fit['stop_reason']}, "
      f"lock-in at iteration {fit['lockin_iter']}, rel_err={fit['rel_err']:.4f}")
print(f"K={fit['K']}  m_min={fit['m_min']}  m_max={fit['m_max']}  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}  "
      f"widths min/median/max = {int(widths.min())}/{int(widths.median())}/{int(widths.max())}")

### 3.1.d′ Reference fit on the original (untransformed) matrix

This is the **same method and config** (`fit_varwidth(A0, CFG, seed=SEED)`) on the raw $A_0 = E_{V^*}^\top$, which is the
experiments' `plain` matrix. `Z0` and `Y0` are used only for comparison: 3.1.e/f (errors), 3.2.e′ (the raw fit's ρ ≥ τ
candidates, **no Gemini judge**) and 3.2.w (feature matching). The fit is cached in `RUN_DIR/fit0.pt`, since it depends on the
variant.

In [ ]:
FIT0_PATH = RUN_DIR / "fit0.pt"
fit0_meta = {"model": MODEL_NAME, "vstar_ids": vstar_ids, "matrix": "plain", "config": CFG.to_dict(), "seed": SEED}
fit0 = torch.load(FIT0_PATH, weights_only=False) if FIT0_PATH.exists() else None
if fit0 is not None and fit0.get("meta") != fit0_meta:
    print(f"{FIT0_PATH} was made with another configuration: re-fitting")
    invalidate(RUN_DIR, ["fit0.pt", "summary.json"])
    fit0 = None
if fit0 is not None:
    print(f"loaded {FIT0_PATH}")
else:
    fit0 = fit_varwidth(A0, CFG, seed=SEED, device=DEVICE, verbose=True)
    fit0["meta"] = fit0_meta
    torch.save(fit0, FIT0_PATH)

Z0 = fit0["Z"].float().cpu()        # (d, k)      raw-space features
Y0 = fit0["Y"].float().cpu()        # (k, |V*|)   sparse coefficients
widths0 = fit0["widths"].long()     # (k,)
t_fit0 = {"elapsed": fit0["elapsed"]}
print(f"\nfitted in {t_fit0['elapsed']:.1f}s: {fit0['n_iter']} iterations, stop={fit0['stop_reason']}, "
      f"lock-in at iteration {fit0['lockin_iter']}, rel_err={fit0['rel_err']:.4f}")
print(f"Z0: {tuple(Z0.shape)}  Y0: {tuple(Y0.shape)}  "
      f"widths min/median/max = {int(widths0.min())}/{int(widths0.median())}/{int(widths0.max())}")

### 3.1.d″ Per-feature widths

The width of feature $i$ is $|S_i| = \#\{t : Y_{i,t}\neq 0\}$. With `fixed` every feature has the same width,
$\lceil s|V^*|\rceil$. With `m1` / `m3` / `rn` the widths vary between $m_{\min}$ and $m_{\max}$ and sum to exactly $K$.
3.2.e″ shows the widths of the selected concept features.

In [ ]:
n_v = len(vstar_ids)
K_budget, m_lo, m_hi = budgets(n_v, CFG)
if CFG.method == "fixed":                       # core._budget_at: fixed uses m_min = m_max = ceil(s n)
    m_lo = m_hi = K_budget // RANK
print(f"method={CFG.method}  |V*|={n_v}  K={K_budget}  [m_min, m_max] = [{m_lo}, {m_hi}]  "
      f"(1% of |V*| = {math.ceil(0.01 * n_v)} tokens)")

width_rows = {}
for name, w in [("whitened", widths), ("original", widths0)]:
    wd = w.double()
    width_rows[name] = {"sum": int(w.sum()), "min": int(w.min()), "q10": wd.quantile(0.1).item(), "median": wd.median().item(),
                        "mean": wd.mean().item(), "q90": wd.quantile(0.9).item(), "max": int(w.max()),
                        "at floor": int((w == m_lo).sum()), "at cap": int((w == m_hi).sum())}
display(pd.DataFrame(width_rows).T)

for name, w in [("whitened", widths), ("original", widths0)]:
    s = pd.Series(w.numpy())
    hist = s.value_counts().sort_index() if s.nunique() <= 12 else s.value_counts(bins=12).sort_index()
    print(f"\n[{name}] number of features per width:")
    print(hist.to_string())

### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature uses its allocated width (3.1.d″; `fixed`: exactly $\lceil s|V^*|\rceil$). Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

Both fits are reported, each against its own data matrix. The two errors are not comparable in absolute terms: whitening
flattens the spectrum, so the transformed error is higher for any rank-$k$ model (the SVD baseline in 3.1.f shows by how much).

The second cell is the optional check of Instructions step 6: $|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ between each fitted factor
and the target mean in X-space ($\tilde{\mathbf m} = $ `m_X`). Concept factors should be among the highest; low values flag generic factors.

In [ ]:
A_cpu, A0_cpu = A.cpu(), A0.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()        # transformed fit, X-space
rel_err0 = (torch.norm(A0_cpu - Z0 @ Y0) / torch.norm(A0_cpu)).item()   # original fit, raw space
nz = Y.abs() > 0
nz0 = Y0.abs() > 0

for name, err, mask in [("transformed", rel_err, nz), ("original", rel_err0, nz0)]:
    tokens_per_feature, features_per_token = mask.sum(1), mask.sum(0)
    print(f"[{name}] relative reconstruction error ||A - ZY||_F / ||A||_F = {err:.4f}")
    print(f"[{name}] tokens per feature : min={tokens_per_feature.min().item()}  "
          f"median={tokens_per_feature.median().item()}  max={tokens_per_feature.max().item()}  total={int(mask.sum())}")
    print(f"[{name}] features per token : mean={features_per_token.float().mean():.2f}  "
          f"max={features_per_token.max().item()}  uncovered tokens={int((features_per_token == 0).sum())}\n")

In [ ]:
# Instructions step 6 (optional check): |cos| of each X-space factor with the target mean m_X.
cos_m = F.cosine_similarity(Z.double(), m_X.unsqueeze(1), dim=0).abs()       # (k,)
cos_m_order = torch.argsort(cos_m, descending=True).tolist()
print(f"|cos(z~_i, m_X)|: median={cos_m.median():.3f}  max={cos_m.max():.3f}  "
      f"(random-direction scale ~ {math.sqrt(2 / (math.pi * D_MODEL)):.3f})")
cos_m_df = pd.DataFrame({
    "feature": cos_m_order[:10],
    "|cos(z~, m_X)|": cos_m[cos_m_order[:10]].numpy(),
    "top tokens": [", ".join(tokenizer.convert_ids_to_tokens(
        [vstar_ids[j] for j in torch.argsort(Y[f].abs(), descending=True)[:8].tolist()])) for f in cos_m_order[:10]],
})
cos_m_df

In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the sparsity constraint costs in reconstruction.

In [ ]:
for name, A_mat, err, mask, Z_mat in [("transformed", A_cpu, rel_err, nz, Z), ("original", A0_cpu, rel_err0, nz0, Z0)]:
    U, S, Vh = torch.linalg.svd(A_mat, full_matrices=False)
    A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
    svd_rel_err = (torch.norm(A_mat - A_svd) / torch.norm(A_mat)).item()

    svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
    smf_nnz = Z_mat.numel() + int(mask.sum())
    print(f"[{name}] rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
    print(f"[{name}] rank-{RANK} Sparse MF     : relative error = {err:.4f}   (nonzero parameters: {smf_nnz:,})")

    # Smallest SVD rank whose error is no worse than Sparse MF's.
    tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
    matching_rank = int((tail <= err).nonzero()[0]) if (tail <= err).any() else len(S)
    print(f"[{name}] SVD reaches the Sparse MF error ({err:.4f}) at rank {matching_rank}\n")
del U, S, Vh, A_svd

---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [ ]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [ ]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

With variable widths a feature narrower than 20 tokens (down to $m_{\min}$) shows the judge all of its tokens, so fewer than 20.

In [ ]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
if SMOKE:
    cand = cand.head(3)  # smoke: judge only the 3 highest-rho candidates
from ember.judge_words import desc_sys_prompt, judge_cache_name, judge_items, token_word_counts

DESC_SYS = desc_sys_prompt(JUDGE_FORMAT, multilingual=False)   # = TOKENS_DESC_SYS for "tokens"
jw_counts = token_word_counts(tokenizer, [s for s, _ in dataset]) if JUDGE_FORMAT == "words" else {}


def judge_input(Ymat: torch.Tensor, f: int, activating_tokens) -> list:
    """What Stage 1 sees for feature f of Ymat: "tokens" = the top activating tokens (as before); "words" = token → source words."""
    if JUDGE_FORMAT == "tokens":
        return list(activating_tokens[:TOP_K_TOKENS])
    mags = Ymat[f].abs()
    top = torch.argsort(mags, descending=True)[:TOP_K_TOKENS]
    top = top[mags[top] > 0].tolist()
    return judge_items([vstar_ids[j] for j in top], lambda t: tokenizer.decode([t]), jw_counts, JUDGE_FORMAT,
                       multilingual=False)


cand["judge_tokens"] = [judge_input(Y, int(f), toks) for f, toks in zip(cand["feature"], cand["activating_tokens"])]
cand[["feature", "ratio_abs", "judge_tokens"]]

### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [ ]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Persisted in RUN_DIR/judge.json, saved after every Gemini call, so a restart re-queries nothing.
# 3.1.d moves judge.json aside whenever it re-fits.
JUDGE_PATH = RUN_DIR / judge_cache_name(JUDGE_FORMAT, "tokens")   # judge.json for the original format
_judge = load_json(JUDGE_PATH, {})
descriptions = {int(f): d for f, d in _judge.get("descriptions", {}).items()}
verdicts = {int(f): tuple(v) for f, v in _judge.get("verdicts", {}).items()}


def save_judge() -> None:
    save_json(JUDGE_PATH, {"descriptions": descriptions, "verdicts": verdicts})

print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [ ]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"], sys_prompt=DESC_SYS)
    save_judge()
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

In [ ]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [ ]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: descriptions[f] for f in cand["feature"].astype(int).tolist() if f in descriptions and f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)
    save_judge()

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

### 3.2.e′ Original fit: the ρ ≥ τ candidates (`F_C0_rho`), no Gemini

**This notebook makes no Gemini calls for the original fit.** The source notebook judged `Y0` with the two-stage judge (`F_C0`).
Here only the mass-ratio filter is applied to `Y0`. Every original feature with $\rho \ge \tau$ is listed with its width and its
top-15 tokens (by $|Y^0_{i,t}|$, tagged [C]oncept / [N]eutral / [B]oth), so you can compare them by eye with the whitened
$F_C$ above. `F_C0_rho` (all ρ ≥ τ candidates of the original fit) replaces `F_C0` in 3.2.w and in the summary. The last line lists
$|\cos(\tilde{\mathbf z}_i, \tilde{\mathbf m})|$ for $i\in F_C$.

In [ ]:
# Original fit (Y0): rho filter only, no judge. F_C0_rho = every original feature with rho >= tau.
N_SHOW0 = 15
stats0 = compute_embedding_stats(Y0.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)
for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"original fit, rho >= {thr:<4}: {(stats0['ratio_abs'] >= thr).sum():3d} / {RANK} features")

cand0 = stats0[stats0["ratio_abs"] >= TAU][["feature", "ratio_abs", "num_concept", "num_neutral", "num_both"]].reset_index(drop=True)
cand0["width"] = cand0["feature"].map(lambda f: int(widths0[int(f)]))
F_C0_rho = sorted(cand0["feature"].astype(int).tolist())
print(f"\noriginal fit: {len(cand0)} candidates pass tau = {TAU}  ->  F_C0_rho = {F_C0_rho}   (whitened fit: F_C = {F_C})\n")

for _, row in cand0.iterrows():
    feat = int(row["feature"])
    mags = Y0[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW0]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
    print(f"=== original feature {feat}  rho={row['ratio_abs']:.2f}  width={int(row['width'])} ===")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

print("|cos(z~_i, m_X)| for i in F_C: " + (", ".join(
    f"{f}: {cos_m[f]:.3f} (rank {cos_m_order.index(f) + 1}/{RANK})" for f in F_C) or "F_C is empty"))
cand0

### 3.2.e″ Widths of the concept features

The widths of the selected features $F_C$ and of every $\rho \ge \tau$ candidate of the whitened fit, compared with the whitened fit's width
distribution (3.1.d″). With `fixed` every width is the same.

In [ ]:
# Widths of the concept features: F_C and all rho >= tau candidates, against the whitened fit's width distribution.
def width_pct(m: int) -> float:
    """Share of the k features that are narrower than m (in %)."""
    return float((widths < m).double().mean() * 100)


print(f"whitened fit widths: min/median/max = {int(widths.min())}/{int(widths.median())}/{int(widths.max())}")
print("F_C widths: " + (", ".join(f"{f}: {int(widths[f])} (wider than {width_pct(int(widths[f])):.0f}% of the {RANK} features)"
                                   for f in F_C) or "F_C is empty"))
cand["width"] = cand["feature"].map(lambda f: int(widths[int(f)]))
cand.sort_values("width", ascending=False)[["feature", "width", "ratio_abs", "is_member", "confidence", "selected"]]

In [ ]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

In [ ]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

In [ ]:
if not F_C:
    print("No feature passed both filters: F_C is empty for this variant.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f}  width={int(widths[feat])} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

In [ ]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat} (width {int(widths[feat])})"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

In [ ]:
TOP_N = 30
E_ref = E_vstar_X.float().cpu()  # X-space rows = transform(E_vstar): the space Z lives in
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

In [ ]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

In [ ]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

### 3.2.g Extended token set: every sentence of the concept's Wikipedia page, $Z$ fixed
$S_C^{ext} = S_C \cup$ all sentences of the article, $S_N^{ext} = S_N \cup$ as many neutral sentences. The new rows go through the **same** transform $T$ fitted in 3.1.c′ (nothing re-estimated), then only a sparse $Y_{ext}$ is fitted against the frozen X-space $Z$. Groups: C / B / N = label in the original $V^*$ (3.2.a), C\* / B\* / N\* = label on the extended sentences; a token gets the first group it matches in the order C, C\*, B, B\*, N, N\*.

Variable width: feature $i$ keeps its 3.1.d width scaled to the larger set, $m_i^{ext} = \mathrm{round}(|S_i|\cdot|V^*_{ext}|/|V^*|)$ (`fixed`: $\lceil s|V^*_{ext}|\rceil$ for every feature, as in the source notebook). The sentences are cached once per concept in `runs/variable_width/Ancient_Rome/ext_sentences.json`, seeded from the source notebook's `runs/whiten/gemma/Ancient_Rome/ext_sentences.json` when it exists (same page revision). This needs the network only if neither file exists.

In [ ]:
# ---- Extended E*: all sentences of the concept's Wikipedia page, Z from 3.1.d fixed, only a sparse Y_ext refit ----
# S_C^ext = S_C + every sentence of the live English Wikipedia article; S_N^ext = S_N + as many neutral sentences
# (one per page of the local wikimedia/wikipedia sample). The new sentences are cached next to RUN_DIR, so re-runs
# use the same page revision and sample.
# Rows go to X-space with the transform T of 3.1.c' exactly as fitted there (same mu_B, W, S_eta: nothing re-estimated),
# then Y_ext = Allocate_m((Z^T Z + lambda I)^{-1} Z^T X_ext^T): the dense Y-step of 3.1.d with Z frozen and the same lambda;
# feature i keeps its 3.1.d width scaled to the larger set, m_i^ext = round(width_i * |V*_ext| / |V*|)
# (fixed: ceil(s |V*_ext|) for every feature, the source notebook's WTA).
import random
import sys


sys.path.insert(0, "experiments/resample_sentences")
from fetch_en_sentences import neutral_sentences, page_paragraphs, split_sentences

EXT_PATH = RUN_DIR.parent / "ext_sentences.json"   # shared by the five variants of this concept
_EXT_SRC = Path("runs/whiten") / MODEL_SLUG / concept_slug / "ext_sentences.json"   # the source notebook's cache
if not EXT_PATH.exists() and _EXT_SRC.exists():   # reuse it: same page revision and neutral sample
    EXT_PATH.write_text(_EXT_SRC.read_text())
EXT_GROUPS = ["C", "C*", "B", "B*", "N", "N*"]

# 1) sentences
ext = load_json(EXT_PATH, None)
if ext is None:
    canon, revid, paras = page_paragraphs(CONCEPT)
    page_sents = split_sentences(paras)
    ext = {"title": canon, "revid": revid, "concept": page_sents,
           "neutral": [r["sentence"] for r in neutral_sentences(
               Path("experiments/resample_sentences/data/wiki_dump_pages.jsonl"), len(page_sents),
               random.Random(SEED), exclude=set(page_sents))]}
    save_json(EXT_PATH, ext)
ext_concept = list(dict.fromkeys([s for s, lbl in dataset if lbl == CONCEPT] + ext["concept"]))
ext_neutral = list(dict.fromkeys([s for s, lbl in dataset if lbl != CONCEPT] + ext["neutral"]))
print(f"'{ext['title']}' rev {ext['revid']}: {len(ext['concept'])} page sentences, {len(ext['neutral'])} new neutral "
      f"({EXT_PATH})\nS_C^ext = {len(ext_concept)} sentences   S_N^ext = {len(ext_neutral)} sentences")

# 2) extended token set (a superset of V*) and its groups
special_ext = set(tokenizer.all_special_ids)


def token_set(sents):
    return {int(t) for s in sents for t in tokenizer(s, add_special_tokens=False)["input_ids"]} - special_ext


c_ids, n_ids = token_set(ext_concept), token_set(ext_neutral)
vstar_ext = sorted(c_ids | n_ids | set(vstar_ids))
codes_ext = np.array([2 if t in c_ids and t in n_ids else 1 if t in c_ids else 0 for t in vstar_ext])
orig_code = dict(zip(vstar_ids, codes.tolist()))


def ext_group(t: int, c_ext: int) -> str:
    """First match in the order C, C*, B, B*, N, N* (C/B/N: label in V*, starred: label on the extended sentences)."""
    o = orig_code.get(t)
    for g, hit in [("C", o == 1), ("C*", c_ext == 1), ("B", o == 2), ("B*", c_ext == 2), ("N", o == 0), ("N*", c_ext == 0)]:
        if hit:
            return g


groups_ext = np.array([ext_group(t, int(c)) for t, c in zip(vstar_ext, codes_ext)])
print(f"|V*_ext| = {len(vstar_ext)}  (|V*| = {len(vstar_ids)}, {len(vstar_ext) - len(vstar_ids)} new)")
display(pd.Series(groups_ext).value_counts().reindex(EXT_GROUPS).fillna(0).astype(int).rename("tokens").to_frame().T)

# 3) X_ext = T(E_{V*_ext}): the 3.1.c' transform, unchanged
ids_ext = torch.tensor(vstar_ext)
E_ext = E.index_select(0, ids_ext.to(E.device)).detach().float().cpu()
X_ext = transform(E_ext)                                                   # (|V*_ext|, d), X-space
pos_ext = {t: j for j, t in enumerate(vstar_ext)}
in_vstar = [pos_ext[t] for t in vstar_ids]
print(f"check: max |X_ext[V*] - E_vstar_X| = {(X_ext[in_vstar] - E_vstar_X.float().cpu()).abs().max():.2e}")

# 4) dense Y-step with Z frozen (same ridge as the fit), then per-feature widths
A_ext = X_ext.T.contiguous().to(DEVICE)                                    # (d, |V*_ext|)
Z_dev = Z.to(DEVICE)
with Timer() as t_ext:
    inv_Z = torch.linalg.inv(Z_dev.T @ Z_dev + torch.eye(Z_dev.shape[1], device=DEVICE) * ridge_used)
    G_ext = A_ext.T @ Z_dev @ inv_Z                                        # (|V*_ext|, k), the G_ layout
    if CFG.method == "fixed":
        m_ext = torch.full((RANK,), math.ceil(SPARSITY * len(vstar_ext)), dtype=torch.long)
    else:
        m_ext = (widths.double() * len(vstar_ext) / len(vstar_ids)).round().long().clamp(1, len(vstar_ext))
    G_ext.mul_(mask_from_row_widths(G_ext.abs().T, m_ext).T.to(G_ext.dtype))   # row i of Y_ext keeps its top m_ext[i]
Y_ext = G_ext.T.float().cpu()                                              # (k, |V*_ext|)
A_ext_cpu = A_ext.cpu()
ext_err = (torch.norm(A_ext_cpu - Z @ Y_ext) / torch.norm(A_ext_cpu)).item()
print(f"fixed-Z fit: {t_ext['elapsed']:.3f}s, widths on V*_ext: {int(m_ext.min())}-{int(m_ext.max())} tokens per feature, "
      f"{int(m_ext.sum())} in total")
print(f"relative error: on X_ext (fixed Z) = {ext_err:.4f}   3.1.d joint fit on A = {rel_err:.4f}")
del A_ext, Z_dev, G_ext, A_ext_cpu, E_ext

In [ ]:
# ---- Rate every token of V*_ext on each concept feature, then list the feature's Y_ext support (X-space, where Z lives) ----
#   <x_t, z~>            inner product of the transformed token row with the feature direction
#   <x_t / ||x_t||, z~>  the same with the token row normalized
#   words                whole words of S_C^ext and S_N^ext the token was cut from (most frequent first)
# z~'s sign is arbitrary (Y absorbs it): oriented as in 3.2.f, so its 3.1 support tokens mostly have positive Y.
# Support table: tokens with Y_ext[feat, t] != 0, by decreasing |Y_ext|; group = first match in C, C*, B, B*, N, N* (3.2.g);
# in V*: the token is in the original token set;  in original support: Y[feat, t] != 0 in the 3.1 fit.
import re
from collections import Counter, defaultdict

SHOW_ROWS = 100   # rows shown per ranking; None shows all |V*_ext| tokens (the full tables are in ext_rating)

WORD_RE = re.compile(r"\w+(?:['’-]\w+)*")
token_words_ext = defaultdict(Counter)  # token id -> Counter of source words
for sent in ext_concept + ext_neutral:
    enc = tokenizer(sent, add_special_tokens=False, return_offsets_mapping=True)
    words = [(m.start(), m.end(), m.group()) for m in WORD_RE.finditer(sent)]
    for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"]):
        for ws, we, w in words:
            if ws < e and s < we:  # token's character span overlaps this word
                token_words_ext[tid][w] += 1
words_ext = [[w for w, _ in token_words_ext[t].most_common()] for t in vstar_ext]

ext_strs = tokenizer.convert_ids_to_tokens(vstar_ext)
x_norms = X_ext.norm(dim=1)
vstar_set = set(vstar_ids)
ext_rating, ext_support = {}, {}
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = X_ext @ z
    orig_sup = {vstar_ids[j] for j in torch.nonzero(y).flatten().tolist()}
    ext_sup = (Y_ext[feat] != 0).numpy()

    # 1) every token of V*_ext, ranked by the two inner products
    # support: first match of (original: in this direction's 3.1 support, extended: in its Y_ext support, none)
    df = pd.DataFrame({
        "token": ext_strs,
        "group": groups_ext,
        "words": words_ext,
        "support": ["original" if t in orig_sup else "extended" if e else "none" for t, e in zip(vstar_ext, ext_sup)],
        "<x_t, z>": dots.numpy(),
        "<x_t / ||x_t||, z>": (dots / x_norms).numpy(),
        "||x_t||": x_norms.numpy(),
        "Y_ext": (sign * Y_ext[feat]).numpy(),
        "in Y_ext support": (Y_ext[feat] != 0).numpy(),
    })
    ext_rating[feat] = df

    print(f"{'=' * 100}\n=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0}   "
          f"|V*_ext| = {len(df)}   Y_ext support = {int(df['in Y_ext support'].sum())} ===\n{'=' * 100}")
    by_group = df.groupby("group")[["<x_t, z>", "<x_t / ||x_t||, z>"]].agg(["mean", "median", "max"])
    by_group.insert(0, "tokens", df["group"].value_counts())
    display(by_group.reindex(EXT_GROUPS).style.format("{:.3f}", subset=by_group.columns[1:]))
    for col in ["<x_t, z>", "<x_t / ||x_t||, z>"]:
        ranked = df.sort_values(col, ascending=False, ignore_index=True)
        ranked.index += 1
        print(f"\n-- all {len(df)} tokens of V*_ext by {col} (top {SHOW_ROWS or len(df)} shown) --")
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(ranked if SHOW_ROWS is None else ranked.head(SHOW_ROWS))

    # 2) the feature's Y_ext support, by decreasing |Y_ext|
    sup = torch.nonzero(Y_ext[feat]).flatten()
    sup = sup[torch.argsort(Y_ext[feat, sup].abs(), descending=True)].tolist()
    sup_df = pd.DataFrame({
        "token": [ext_strs[j] for j in sup],
        "group": groups_ext[sup],
        "words": [words_ext[j] for j in sup],
        "Y_ext": (sign * Y_ext[feat, sup]).numpy(),
        "in V*": [vstar_ext[j] in vstar_set for j in sup],
        "in original support": [vstar_ext[j] in orig_sup for j in sup],
        "<x_t, z>": dots[sup].numpy(),
        "<x_t / ||x_t||, z>": (dots[sup] / x_norms[sup]).numpy(),
    }, index=pd.RangeIndex(1, len(sup) + 1, name="rank"))
    ext_support[feat] = sup_df

    lost = sorted(orig_sup - {vstar_ext[j] for j in sup})
    counts = "  ".join(f"{g}: {int((sup_df['group'] == g).sum())}" for g in EXT_GROUPS)
    print(f"\n-- Y_ext support: {len(sup_df)} tokens [{counts}] | in V*: {int(sup_df['in V*'].sum())} | "
          f"in original support: {int(sup_df['in original support'].sum())}/{len(orig_sup)} --")
    if lost:
        print(f"original support tokens no longer in the support ({len(lost)}): {tokenizer.convert_ids_to_tokens(lost)}")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(sup_df.style.format({"Y_ext": "{:+.4f}", "<x_t, z>": "{:.2f}", "<x_t / ||x_t||, z>": "{:.3f}"}))

---
### 3.2.w Original vs transformed: matched-feature overlap

Do the two fits find the same token groups? Each feature is compared through its **support**, the tokens it uses:
$S^0_i = \{t : Y^0_{i,t}\neq 0\}$ (original fit) and $S_j = \{t : Y_{j,t}\neq 0\}$ (transformed fit).

- Both fits use the same `CFG`, so both have $K$ non-zeros in total. With `fixed` every row has exactly $\lceil s|V^*|\rceil$;
  with `m1` / `m3` / `rn` the per-feature widths $|S^0_i|$, $|S_j|$ vary (and are not paired across fits). Jaccard normalizes for that.
- $J_{ij} = |S^0_i\cap S_j| \,/\, |S^0_i\cup S_j|$ for all $k\times k$ pairs. Feature indices are arbitrary in each fit, so
  features are paired by the one-to-one matching with the largest total Jaccard (Hungarian algorithm,
  `scipy.optimize.linear_sum_assignment`). `best Jaccard` is the best partner without the one-to-one constraint.
- Directions: a transformed feature lives in X-space. `T.unmap` ($\tilde{\mathbf z}\,S_\eta^{-\gamma/2}W^{-1}$) maps it to the raw space,
  where it is compared with $\mathbf z^0_i$ by $|\cos|$ (the sign of a feature is arbitrary).
- **Original-fit concept set.** The original fit is not judged in this notebook. Where the source notebook used the judged `F_C0`, the columns and lists below use `F_C0_rho` (all ρ ≥ τ candidates of the original fit, 3.2.e′). That set is larger than a judged $F_{C0}$ would be.

In [ ]:
from scipy.optimize import linear_sum_assignment

S0 = (Y0 != 0).numpy()                    # (k, |V*|) supports, original fit
S1 = (Y != 0).numpy()                     # (k, |V*|) supports, transformed fit
print(f"widths |S0_i| (original): min/median/max = {int(S0.sum(1).min())}/{int(np.median(S0.sum(1)))}/{int(S0.sum(1).max())}   "
      f"|S_j| (transformed): {int(S1.sum(1).min())}/{int(np.median(S1.sum(1)))}/{int(S1.sum(1).max())}   "
      f"total: {int(S0.sum())} / {int(S1.sum())}")

inter = S0.astype(np.int64) @ S1.T.astype(np.int64)            # (k, k): |S0_i ∩ S_j|
union = S0.sum(1)[:, None] + S1.sum(1)[None, :] - inter
J = inter / np.maximum(union, 1)

rows0, cols1 = linear_sum_assignment(-J)                       # maximize total Jaccard
match0 = dict(zip(rows0.tolist(), cols1.tolist()))             # original i -> transformed j
match1 = {j: i for i, j in match0.items()}                     # transformed j -> original i

Z_back = T.unmap(Z.T).T.float()                                # (d, k): transformed features in the raw space
cos_map = F.cosine_similarity(Z0[:, rows0], Z_back[:, cols1], dim=0).abs().numpy()

match_df = pd.DataFrame({
    "orig feature": rows0,
    "transformed feature": cols1,
    "|S0|": S0.sum(1)[rows0],
    "|S|": S1.sum(1)[cols1],
    "|S0∩S|": inter[rows0, cols1],
    "Jaccard": J[rows0, cols1],
    "best Jaccard": J.max(1)[rows0],
    "best partner": J.argmax(1)[rows0],
    "|cos(z0, unmap(z~))|": cos_map,
    "orig in F_C0_rho": [i in F_C0_rho for i in rows0.tolist()],   # original fit: rho >= tau candidates (not judged)
    "transformed in F_C": [j in F_C for j in cols1.tolist()],
}).sort_values("Jaccard", ascending=False, ignore_index=True)

mj = match_df["Jaccard"]
print("matched Jaccard quantiles: " + "  ".join(f"q{int(q * 100)}={mj.quantile(q):.3f}" for q in (0, .1, .25, .5, .75, .9, 1)))
print(f"pairs with Jaccard >= 0.5: {(mj >= 0.5).sum()} / {RANK}   >= 0.8: {(mj >= 0.8).sum()} / {RANK}   "
      f"matched partner = best partner: {(match_df['transformed feature'] == match_df['best partner']).sum()} / {RANK}")
print(f"|cos| of mapped directions: median={match_df['|cos(z0, unmap(z~))|'].median():.3f}   "
      f"Spearman vs Jaccard = {match_df['Jaccard'].corr(match_df['|cos(z0, unmap(z~))|'], method='spearman'):.3f}")
print("\nhistogram of matched Jaccard:")
print(pd.cut(mj, bins=np.linspace(0, 1, 11), include_lowest=True).value_counts().sort_index().to_string())
with pd.option_context("display.max_rows", None):
    display(match_df)

#### Concept features in both fits

For each transformed concept feature $f\in F_C$ and each original candidate $f_0\in$ `F_C0_rho` (the original fit's ρ ≥ τ candidates; no judge): its matched partner in
the other fit, their Jaccard, and whether the partner is itself a concept feature of its own fit. The token table splits
the union of the two supports into *both* / *only original* / *only transformed*, with labels (C / N / B) and the signed
$Y^0$ and $Y$ values, sorted by the larger magnitude within each group.

In [ ]:
def pair_table(i: int, j: int) -> pd.DataFrame:
    """Union of S0_i and S_j: membership, label, Y0[i, t], Y[j, t]; grouped, then sorted by magnitude."""
    cols = np.where(S0[i] | S1[j])[0]
    where = np.where(S0[i, cols] & S1[j, cols], "both", np.where(S0[i, cols], "only original", "only transformed"))
    df = pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens([vstar_ids[t] for t in cols]),
        "label": [LABEL_TAG[codes[t]] for t in cols],
        "in": where,
        f"Y0[{i}]": Y0[i, cols].numpy(),
        f"Y[{j}]": Y[j, cols].numpy(),
    })
    group = df["in"].map({"both": 0, "only original": 1, "only transformed": 2})
    mag = np.maximum(df[f"Y0[{i}]"].abs(), df[f"Y[{j}]"].abs())
    return df.assign(_g=group, _m=mag).sort_values(["_g", "_m"], ascending=[True, False]) \
             .drop(columns=["_g", "_m"]).reset_index(drop=True)


pairs = {}
for f in F_C:
    i = match1[f]
    pairs.setdefault((i, f), []).append(
        f"transformed concept feature {f} (width {int(widths[f])}): matched original feature {i} is {'IN' if i in F_C0_rho else 'NOT in'} F_C0_rho (rho >= tau) "
        f"(best-Jaccard original partner: {int(J[:, f].argmax())}, J={J[:, f].max():.3f})")
for f0 in F_C0_rho:   # original fit: rho >= tau candidates instead of the judged F_C0
    j = match0[f0]
    pairs.setdefault((f0, j), []).append(
        f"original rho >= tau candidate {f0} (width {int(widths0[f0])}): matched transformed feature {j} is {'IN' if j in F_C else 'NOT in'} F_C "
        f"(best-Jaccard transformed partner: {int(J[f0].argmax())}, J={J[f0].max():.3f})")

print(f"F_C (transformed) = {F_C}   F_C0_rho (original, rho >= tau) = {F_C0_rho}   "
      f"F_C mapped into F_C0_rho: {sum(match1[f] in F_C0_rho for f in F_C)} / {len(F_C)}\n")
if not pairs:
    print("F_C and F_C0_rho are both empty: nothing to compare.")

for (i, j), notes in sorted(pairs.items()):
    tab = pair_table(i, j)
    n_in = tab["in"].value_counts()
    cos_ij = F.cosine_similarity(Z0[:, i], Z_back[:, j], dim=0).abs().item()
    print(f"{'=' * 100}\n=== original {i} <-> transformed {j}   Jaccard = {J[i, j]:.3f}   "
          f"|cos(z0, unmap(z~))| = {cos_ij:.3f}\n{'=' * 100}")
    for note in notes:
        print("  " + note)
    print(f"  both: {n_in.get('both', 0)}   only original: {n_in.get('only original', 0)}   "
          f"only transformed: {n_in.get('only transformed', 0)}")
    display(pd.crosstab(tab["in"], tab["label"]))
    with pd.option_context("display.max_rows", None):
        display(tab)

#### Run summary → `RUN_DIR/summary.json`

This writes the configuration (variant and `VWConfig`), $|V^*|$, the relative errors and convergence of both fits, the per-feature
widths of both fits, $\rho_{\max}$, $F_C$ with each feature's width, `F_C0_rho` (the original fit's ρ ≥ τ candidates, **not judged**),
each $F_C$ feature's matched original partner (Jaccard, whether the partner is in `F_C0_rho`), and the number of concept tokens the
$F_C$ features touch (what Step 3.3 would edit; Step 3.3 is not part of this notebook).

In [ ]:
def write_summary(**extra) -> dict:
    """Write RUN_DIR/summary.json from the current state; `extra` adds or overrides fields."""
    eligible_cols = ((Y[F_C].abs() > 0).any(dim=0).numpy() & (codes == 1)) if F_C else np.zeros(len(vstar_ids), bool)
    summary = {
        "config": {"model": MODEL_NAME, "concept": CONCEPT, "variant": VARIANT, "vwconfig": CFG.to_dict(), "seed": SEED,
                   "background": BACKGROUND, "target": TARGET, "gamma": GAMMA, "eta": ETA, "eps_rel": EPS_REL,
                   "eps": float(bw.eps), "rank": RANK, "smoke": SMOKE, "n_target": len(target_ids)},
        "n_vstar": len(vstar_ids),
        "rel_err": {"original": rel_err0, "transformed": rel_err},
        "fit": {name: {"n_iter": f["n_iter"], "stop_reason": f["stop_reason"], "lockin_iter": f["lockin_iter"],
                       "elapsed": f["elapsed"], "K": f["K"], "m_min": f["m_min"], "m_max": f["m_max"]}
                for name, f in [("transformed", fit), ("original", fit0)]},
        "widths": widths.tolist(),
        "widths_original": widths0.tolist(),
        "rho_max": float(stats["ratio_abs"].max()),
        "rho_max_original": float(stats0["ratio_abs"].max()),
        "n_F_C": len(F_C),
        "F_C": F_C,
        "tokens_per_feature": {f: int(widths[f]) for f in F_C},
        "F_C0_rho": F_C0_rho,   # original fit: rho >= tau candidates (no Gemini judge in this notebook)
        "F_C_matched": [{"feature": f, "width": int(widths[f]), "partner": match1[f], "partner_width": int(widths0[match1[f]]),
                         "jaccard": float(J[match1[f], f]), "partner_in_F_C0_rho": match1[f] in F_C0_rho,
                         "cos_m": float(cos_m[f])} for f in F_C],
        "F_C0_rho_matched": [{"feature": f0, "width": int(widths0[f0]), "partner": match0[f0],
                              "jaccard": float(J[f0, match0[f0]]), "partner_in_F_C": match0[f0] in F_C} for f0 in F_C0_rho],
        "matched_jaccard_median": float(np.median(J[rows0, cols1])),
        "n_edit": int(eligible_cols.sum()),   # concept-only tokens in the support of F_C (what Step 3.3 would edit)
        **extra,
    }
    save_json(RUN_DIR / "summary.json", summary)
    print(f"wrote {RUN_DIR / 'summary.json'}: F_C={F_C} F_C0_rho={F_C0_rho} n_edit={summary['n_edit']} "
          f"rel_err={rel_err:.4f} (orig {rel_err0:.4f}) rho_max={summary['rho_max']:.2f}")
    return summary


summary = write_summary()